# 08 XGBoost Tuning
Use RandomizedSearchCV with StratifiedKFold and average precision as the development objective.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import PROCESSED_DATA_DIR, RANDOM_STATE
from src.preprocessing import make_xgboost_preprocessor
from src.xgboost_model import baseline_xgboost

X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
preprocessor = make_xgboost_preprocessor(X_train)
pipeline = Pipeline([("preprocess", preprocessor), ("model", baseline_xgboost())])
search = RandomizedSearchCV(
    pipeline,
    {"model__n_estimators": [150, 250], "model__max_depth": [3, 4], "model__learning_rate": [0.03, 0.05], "model__min_child_weight": [1, 3]},
    n_iter=2,
    scoring="average_precision",
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
search.fit(X_train, y_train)
print("Best development PR-AUC:", search.best_score_)
print("Best parameters:", search.best_params_)
search.best_estimator_